# Ejemplo 03 (versión instructiva): baño retardado no Hermítico regional en el borde superior

Este notebook se mantiene **simple e instructivo**: comparamos solo dos casos
\(\gamma_y=0\) y \(\gamma_y\neq 0\), y nos enfocamos en comparar la **dinámica** de corrientes y densidades.


## 1) Parámetros mínimos (editar aquí)
Se dejan pocas opciones para facilitar lectura y ejecución top-to-bottom.


In [ ]:
# Geometría fija pequeña (rápida para demostración)
Nx, Ny = 6, 2

# Modelo electrónico
Nσ = 2
N_orb = 1
γ = 1.0
γso = 0.50 + 0.0im
μ = 0.0
Bz = 0.15

# Leads
β = 33.0
N_λ1 = 49
N_λ2 = 20

# Baño retardado superior
gamma0 = 0.20
gammay_caseA = 0.0
gammay_caseB = 0.10

# Tiempo de simulación
tspan = (0.0, 18.0)
dt_save = 1.0
reltol = 1e-6
abstol = 1e-8

# Paleta sobria
PALETTE = Dict(
    :navy => "#1f3b73",
    :teal => "#2a9d8f",
    :cyan => "#5bc0be",
    :gold => "#e9c46a",
    :orange => "#f4a261",
    :lightbg => "#f7f8fa",
)

using PyPlot
plt.rcParams["axes.facecolor"] = PALETTE[:lightbg]
plt.rcParams["figure.facecolor"] = "white"
plt.rcParams["axes.grid"] = true
plt.rcParams["grid.alpha"] = 0.25

println("Nx=$(Nx), Ny=$(Ny), tspan=$(tspan), dt=$(dt_save)")
println("Comparación: gammay=$(gammay_caseA) vs gammay=$(gammay_caseB)")


## 2) Imports y utilidades de geometría


In [ ]:
using Pkg
Pkg.activate(joinpath(dirname(dirname(@__DIR__))))

using TDNEGF
using DifferentialEquations
using LinearAlgebra
using Statistics

@inline local_index(x::Int, y::Int, Ny::Int) = (x - 1) * Ny + y

@inline function local_subrange(site::Int, N_loc::Int)
    a = (site - 1) * N_loc + 1
    b = site * N_loc
    return a:b
end

function top_edge_coupled_indices(; Nx::Int, Ny::Int, N_loc::Int)
    top_sites = [local_index(x, Ny, Ny) for x in 1:Nx]
    idx_couple = Int[]
    for s in top_sites
        append!(idx_couple, collect(local_subrange(s, N_loc)))
    end
    return top_sites, idx_couple
end

function site_scalar_to_grid(values::AbstractVector, Nx::Int, Ny::Int)
    @assert length(values) == Nx * Ny
    [values[local_index(x, y, Ny)] for y in Ny:-1:1, x in 1:Nx]
end


## 3) Geometría y sitios acoplados al baño
Mostramos claramente qué borde está acoplado al baño retardado (fila superior).


In [ ]:
N_sites = Nx * Ny
N_loc = Nσ * N_orb

top_sites, idx_couple = top_edge_coupled_indices(; Nx, Ny, N_loc)
bottom_sites = [local_index(x, 1, Ny) for x in 1:Nx]

site_class = zeros(Float64, N_sites)
site_class[top_sites] .= 1.0
site_class[bottom_sites] .= -1.0

fig, ax = plt.subplots(figsize=(6.0, 3.0))
img = ax.imshow(site_scalar_to_grid(site_class, Nx, Ny), cmap="coolwarm", vmin=-1, vmax=1, aspect="auto")
ax.set_title(raw"$\mathrm{Sitios\ acoplados:\ borde\ superior}$")
ax.set_xlabel(raw"$x$")
ax.set_ylabel(raw"$y\,(\mathrm{arriba\ en\ la\ parte\ superior})$")
plt.colorbar(img, ax=ax, label=raw"$+1:\mathrm{top},\ -1:\mathrm{bottom}$")
plt.show()

println("top_sites   = ", top_sites)
println("idx_couple  = ", idx_couple)
println("size local  = ", length(idx_couple))


## 4) Construcción del sistema (arquitectura sin cambios)
Se mantiene el baño extra en la ruta experimental regional (`ExtraBathSpec` + `ExperimentalBlockRHSParams`).


In [ ]:
function add_uniform_onsite!(H::AbstractMatrix{ComplexF64}; Nx::Int, Ny::Int, N_loc::Int, μ::Float64, Bz::Float64)
    σz = ComplexF64[1 0; 0 -1]
    onsite = (-μ) * Matrix{ComplexF64}(I, N_loc, N_loc) + Bz * σz
    for site in 1:(Nx * Ny)
        r = local_subrange(site, N_loc)
        H[r, r] .+= onsite
    end
    return H
end

function build_two_terminal_blocks(p_model::ModelParamsTDNEGF; β::Float64, N_λ1::Int, N_λ2::Int)
    Rλ, zλ = load_poles_square(N_λ1, N_λ2)
    Σᴸ_nλ = build_Σᴸ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb, p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)
    Σᴳ_nλ = build_Σᴳ_nλ(Rλ, zλ, p_model.Ny, p_model.Nσ, p_model.N_orb, p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)
    χ_nλ = build_χ_nλ(zλ, p_model.Ny, p_model.Nσ, p_model.N_orb, p_model.N_λ1, p_model.N_λ2; β=β, γ=1.0)

    ξ_anR = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb; xcol=p_model.Nx, y_coup=1:p_model.Ny)
    ξ_anL = build_ξ_an(p_model.Nx, p_model.Ny, p_model.Nσ, p_model.N_orb; xcol=1, y_coup=1:p_model.Ny)

    left_block = SelfEnergyBlock(:left, p_model.Nc, p_model.N_λ1, p_model.N_λ2, Σᴸ_nλ, Σᴳ_nλ, χ_nλ, ξ_anL)
    right_block = SelfEnergyBlock(:right, p_model.Nc, p_model.N_λ1, p_model.N_λ2, Σᴸ_nλ, Σᴳ_nλ, χ_nλ, ξ_anR)
    return [left_block, right_block]
end

function build_top_bath_ΣR(; n_coupled_sites::Int, N_loc::Int, gamma0::Float64, gammay::Float64)
    @assert N_loc == 2
    @assert gamma0 >= 0.0
    if gamma0 < abs(gammay)
        @warn "gamma0 < |gammay|: puede aparecer ganancia efectiva."
    end
    σy = ComplexF64[0 -im; im 0]
    Σ_site = -1im .* (gamma0 .* Matrix{ComplexF64}(I, N_loc, N_loc) .+ gammay .* σy)
    ΣR_loc = kron(Matrix{ComplexF64}(I, n_coupled_sites, n_coupled_sites), Σ_site)
    return Σ_site, ΣR_loc
end

function embed_local_block(idx_couple::Vector{Int}, block_local::AbstractMatrix{ComplexF64}, Nglobal::Int)
    Σ_global = zeros(ComplexF64, Nglobal, Nglobal)
    Σ_global[idx_couple, idx_couple] .= block_local
    return Σ_global
end

function make_case(; gammay::Float64)
    p_model = ModelParamsTDNEGF(Nx=Nx, Ny=Ny, Nσ=Nσ, N_orb=N_orb, Nα=2, N_λ1=N_λ1, N_λ2=N_λ2)

    H_device = build_H_ab(; Nx, Ny, Nσ, N_orb, γ=γ, γso=γso)
    add_uniform_onsite!(H_device; Nx=Nx, Ny=Ny, N_loc=p_model.N_loc, μ=μ, Bz=Bz)
    p_model.H_ab .= H_device
    p_model.H0_ab .= H_device

    blocks = build_two_terminal_blocks(p_model; β=β, N_λ1=N_λ1, N_λ2=N_λ2)
    Δ_blocks = ComplexF64[0.5 + 0.0im, -0.5 + 0.0im]

    top_sites, idx_couple = top_edge_coupled_indices(; Nx=Nx, Ny=Ny, N_loc=p_model.N_loc)
    Σ_site, ΣR_loc = build_top_bath_ΣR(; n_coupled_sites=length(top_sites), N_loc=p_model.N_loc, gamma0=gamma0, gammay=gammay)
    ΣR_top_global = embed_local_block(idx_couple, ΣR_loc, size(H_device, 1))

    top_bath = TDNEGF.ExtraBathSpec(:nonhermitian_retarded, :top_edge_nonhermitian, idx_couple, ΣR_loc, true)
    p_blocks = ExperimentalBlockRHSParams(p_model.H_ab, blocks, Δ_blocks, p_model, [top_bath])

    u0 = zeros(ComplexF64, p_blocks.dims_ρ_ab[1]^2 + p_blocks.aux_layout.total_size)

    return (; p_model, p_blocks, u0, top_sites, idx_couple, ΣR_top_global, gammay)
end

caseA = make_case(; gammay=gammay_caseA)
caseB = make_case(; gammay=gammay_caseB)

println("A: gammay=$(caseA.gammay), B: gammay=$(caseB.gammay)")
println("gamma0 >= |gammay_caseB| ? ", gamma0 >= abs(gammay_caseB))


## 5) Dinámica para ambos casos
Ejecutamos la misma dinámica para A y B, y luego comparamos observables.


In [ ]:
function run_case(case_setup)
    save_times = collect(range(tspan[1], tspan[2]; step=dt_save))
    prob = ODEProblem(eom_tdnegf_blocks!, case_setup.u0, tspan, case_setup.p_blocks)
    sol = solve(prob, Vern7(); adaptive=true, saveat=save_times, dense=false, reltol=reltol, abstol=abstol)

    obs = ObservablesTDNEGF(case_setup.p_model; N_tmax=length(sol.t), N_leads=length(case_setup.p_blocks.blocks))
    obs.t = sol.t

    for (it, ut) in enumerate(sol.u)
        obs.idx = it
        ptr = pointer_blocks(ut, case_setup.p_blocks.dims_ρ_ab, case_setup.p_blocks.aux_layout)
        obs_n_i!(ptr, case_setup.p_model, obs)
        obs_σ_i!(ptr, case_setup.p_model, obs)
        obs_Ixα!(ptr, case_setup.p_blocks, obs)
    end

    return sol, obs
end

solA, obsA = run_case(caseA)
solB, obsB = run_case(caseB)

println("N_t A = $(length(obsA.t)), N_t B = $(length(obsB.t))")


## 6) Corrientes de carga en los leads
Diagnóstico principal de transporte: comparar \(I_L(t)\) e \(I_R(t)\) entre `gammay=0` y `gammay\neq 0`.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(11.5, 4.0), sharey=true, constrained_layout=true)

axs[1].plot(obsA.t, obsA.Iα[1, :], color=PALETTE[:navy], lw=2.0, label=raw"$I_L$")
axs[1].plot(obsA.t, obsA.Iα[2, :], color=PALETTE[:teal], lw=2.0, ls="--", label=raw"$I_R$")
axs[1].set_title(raw"$\mathrm{Caso\ A}:\ \gamma_y=0$")
axs[1].set_xlabel(raw"$t$")
axs[1].set_ylabel(raw"$I_{\alpha}(t)$")
axs[1].legend(frameon=false)

axs[2].plot(obsB.t, obsB.Iα[1, :], color=PALETTE[:orange], lw=2.0, label=raw"$I_L$")
axs[2].plot(obsB.t, obsB.Iα[2, :], color=PALETTE[:cyan], lw=2.0, ls="--", label=raw"$I_R$")
axs[2].set_title(raw"$\mathrm{Caso\ B}:\ \gamma_y\neq 0$")
axs[2].set_xlabel(raw"$t$")
axs[2].legend(frameon=false)

plt.show()


## 7) Corrientes de spin en los leads
Se comparan componentes \(I^x, I^y, I^z\) para cada lead y cada caso.


In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(12.0, 4.2), sharey=true, constrained_layout=true)
comps = [(1, raw"$I^x$", PALETTE[:navy]), (2, raw"$I^y$", PALETTE[:teal]), (3, raw"$I^z$", PALETTE[:gold])]

for c, lbl, col in comps
    axs[1].plot(obsA.t, obsA.Iαx[1, c, :], color=col, lw=1.8, label=lbl * raw"$_L$")
    axs[1].plot(obsA.t, obsA.Iαx[2, c, :], color=col, lw=1.8, ls="--", alpha=0.85, label=lbl * raw"$_R$")
    axs[2].plot(obsB.t, obsB.Iαx[1, c, :], color=col, lw=1.8, label=lbl * raw"$_L$")
    axs[2].plot(obsB.t, obsB.Iαx[2, c, :], color=col, lw=1.8, ls="--", alpha=0.85, label=lbl * raw"$_R$")
end

axs[1].set_title(raw"$\mathrm{Caso\ A}:\ \gamma_y=0$")
axs[2].set_title(raw"$\mathrm{Caso\ B}:\ \gamma_y\neq 0$")
for ax in axs
    ax.set_xlabel(raw"$t$")
    ax.legend(frameon=false, fontsize=8, ncol=2)
end
axs[1].set_ylabel(raw"$I^s_{\alpha}(t)$")
plt.show()


## 8) Densidades finales (comparación simple)
Como apoyo visual, mostramos densidad de carga final \(n_i(t_f)\) y componente de spin \(s^z_i(t_f)\) para A y B.


In [ ]:
function final_maps(obs)
    n_final = vec(obs.n_i[:, end])
    sz_final = vec(obs.σx_i[:, 3, end])
    return site_scalar_to_grid(n_final, Nx, Ny), site_scalar_to_grid(sz_final, Nx, Ny)
end

nA, szA = final_maps(obsA)
nB, szB = final_maps(obsB)

fig, axs = plt.subplots(2, 2, figsize=(10.0, 6.0), constrained_layout=true)

im1 = axs[1,1].imshow(nA, cmap="viridis", aspect="auto")
axs[1,1].set_title(raw"$n_i(t_f)$ Caso A")
axs[1,1].set_xlabel(raw"$x$"); axs[1,1].set_ylabel(raw"$y$")
plt.colorbar(im1, ax=axs[1,1], fraction=0.046)

im2 = axs[1,2].imshow(nB, cmap="viridis", aspect="auto")
axs[1,2].set_title(raw"$n_i(t_f)$ Caso B")
axs[1,2].set_xlabel(raw"$x$"); axs[1,2].set_ylabel(raw"$y$")
plt.colorbar(im2, ax=axs[1,2], fraction=0.046)

vmax = max(maximum(abs.(szA)), maximum(abs.(szB))) + 1e-12
im3 = axs[2,1].imshow(szA, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
axs[2,1].set_title(raw"$s^z_i(t_f)$ Caso A")
axs[2,1].set_xlabel(raw"$x$"); axs[2,1].set_ylabel(raw"$y$")
plt.colorbar(im3, ax=axs[2,1], fraction=0.046)

im4 = axs[2,2].imshow(szB, cmap="RdBu_r", vmin=-vmax, vmax=vmax, aspect="auto")
axs[2,2].set_title(raw"$s^z_i(t_f)$ Caso B")
axs[2,2].set_xlabel(raw"$x$"); axs[2,2].set_ylabel(raw"$y$")
plt.colorbar(im4, ax=axs[2,2], fraction=0.046)

plt.show()


## 9) Conclusiones
- Este ejemplo queda intencionalmente **minimalista**: solo dos casos y comparación dinámica de corrientes/densidades.
- La parte de estudios complejos (barridos extensos, análisis más profundo de modos, etc.) se deja fuera para proyectos específicos.
- Se preserva el alcance retarded-only y la arquitectura experimental regional sin cambios.
